# Model Training — Final
### Machine Learning-Based IDS — CICIDS2017 → Suricata Deployment

**PURPOSE OF THIS NOTEBOOK:**
Takes the preprocessed dataset from data_preprocessing_final.ipynb and trains
the Random Forest classifier. Also handles per-class threshold tuning, evaluation,
and saving all deployment artifacts.

**INPUTS (from data_preprocessing_final.ipynb):**
- `X_train.csv` / `X_test.csv` — 19 Suricata-aligned features
- `y_train.csv` / `y_test.csv` — integer-encoded class labels
- `label_encoder.pkl` — converts predictions back to class names
- `feature_contract.pkl` — ordered list of 19 feature names

**CLASSIFIER:** Random Forest with `class_weight='balanced'`

**PRIMARY METRIC:** Macro F1 — equal weight across all 6 classes
(not weighted F1, which would be dominated by BENIGN's 83% share)

**OUTPUTS (saved at end of notebook):**
| File | Purpose |
|------|---------|
| random_forest_ids.pkl | The trained model (32MB, 200 trees) |
| class_thresholds.json | Per-class confidence thresholds for detector.py |
| training_results.json | Full metrics summary |

## 1. Libraries

In [ ]:
import json
import joblib       # Load/save model artifacts
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,      # Per-class precision/recall/F1
    confusion_matrix,           # Full N×N confusion table
    f1_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,     # Plots confusion matrix with colour coding
)

sns.set(style='darkgrid')
print('Libraries loaded successfully.')

## 2. Configuration

In [ ]:
RANDOM_STATE = 42   # Must match the value used in preprocessing to ensure
                    # the same train/test split is reproduced

# ── Random Forest hyperparameters ─────────────────────────────────────────────
RF_PARAMS = {
    'n_estimators'    : 200,          # Number of decision trees in the forest
                                      # More trees = more stable predictions but slower training
                                      # 200 was chosen as a balance — beyond 200 the gain is marginal
    'class_weight'    : 'balanced',   # Automatically upweight minority classes during training
                                      # Equivalent to SMOTE but without data modification
    'max_depth'       : None,         # Trees grow fully until all leaves are pure
                                      # Unrestricted depth is standard for RF — the ensemble
                                      # prevents overfitting even with deep trees
    'min_samples_leaf': 5,            # Each leaf must contain at least 5 samples
                                      # Prevents the model from memorising single noisy flows
    'max_features'    : 'sqrt',       # At each split, consider sqrt(19) ≈ 4 features
                                      # Standard setting for classification — creates diversity
                                      # across trees by preventing them all from using the same features
    'n_jobs'          : -1,           # Use all available CPU cores (12 threads on our server)
                                      # Parallel tree building — 200 trees in ~202 seconds
    'random_state'    : RANDOM_STATE, # Fixed seed for reproducibility
    'verbose'         : 1,            # Print progress during training (shows tree count)
}

# Classes below this recall after tuning are flagged as potentially too weak
RECALL_WARNING_THRESHOLD = 0.60

print('Configuration set.')
print(f'RF params: {RF_PARAMS}')

## 3. Load Preprocessed Data

In [ ]:
# Load the preprocessed feature matrices saved by data_preprocessing_final.ipynb
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()   # squeeze() converts 1-column DataFrame → Series
y_test  = pd.read_csv('y_test.csv').squeeze()

# Load supporting artifacts
le               = joblib.load('label_encoder.pkl')    # Class name ↔ integer mapping
feature_contract = joblib.load('feature_contract.pkl') # Ordered list of 19 feature names

# CRITICAL: enforce feature order from the contract
# CSV loading can sometimes reorder columns — this guarantees the model
# receives features in the exact order it was trained on
X_train = X_train[feature_contract]
X_test  = X_test[feature_contract]

print(f'X_train: {X_train.shape[0]:,} rows x {X_train.shape[1]} features')
print(f'X_test:  {X_test.shape[0]:,} rows x {X_test.shape[1]} features')
print(f'Classes: {list(le.classes_)}')
# Expected: 2,017,852 train rows, 504,463 test rows, 6 classes

## 4. Train Random Forest

Training 200 trees on 2,017,852 samples using 12 CPU threads.
Expected training time: ~202 seconds (3–4 minutes).

The `verbose=1` setting will print progress like:
`[Parallel(n_jobs=-1)]: Done  42 out of 200 | elapsed:  36.7s remaining: 135.7s`

In [ ]:
import time

rf = RandomForestClassifier(**RF_PARAMS)

print('Training Random Forest...')
t0 = time.time()
rf.fit(X_train, y_train)   # This is the actual training — all 200 trees built here
elapsed = time.time() - t0

print(f'\nTraining complete in {elapsed:.1f}s')
print(f'Trees: {rf.n_estimators}')
print(f'Features used: {rf.n_features_in_}')   # Should be 19

## 5. Predict

In [ ]:
# Run predictions on the held-out test set (504,463 flows)
y_pred       = rf.predict(X_test)          # Hard prediction: the argmax class per flow
y_pred_proba = rf.predict_proba(X_test)    # Soft prediction: probability for each class
                                            # Shape: (504463, 6) — one column per class

print(f'Predictions generated for {len(y_pred):,} test samples.')
# y_pred_proba is used in Step 11 (threshold tuning) and Step 10 (calibration check)

## 6. Classification Report

**Why macro F1 is the primary metric:**
Weighted F1 is dominated by BENIGN (83% of data) and would show ~0.997
even if the model completely failed on minority classes. Macro F1 weights
each of the 6 classes equally — a weak Bot or Web Attack class pulls the
macro F1 down regardless of how well BENIGN is classified.

**Baseline results (before threshold tuning):**
- Macro F1 ≈ 0.771
- Bot precision ≈ 0.290 (too many BENIGN flows flagged as Bot)
- Web Attack precision ≈ 0.125 (too many BENIGN flows flagged as Web Attack)

Per-class threshold tuning in Step 12 improves this to macro F1 ≈ 0.810.

In [ ]:
print('=' * 65)
print('CLASSIFICATION REPORT (before threshold tuning)')
print('=' * 65)
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    digits=3
))

macro_f1     = f1_score(y_test, y_pred, average='macro')
macro_prec   = precision_score(y_test, y_pred, average='macro')
macro_recall = recall_score(y_test, y_pred, average='macro')
weighted_f1  = f1_score(y_test, y_pred, average='weighted')

print(f'Headline metric  →  Macro F1:     {macro_f1:.4f}')
print(f'Macro Precision: {macro_prec:.4f}')
print(f'Macro Recall:    {macro_recall:.4f}')
print(f'Weighted F1:     {weighted_f1:.4f}  (biased toward BENIGN — use macro F1 as primary)')
# Expected: macro F1 ≈ 0.771 before threshold tuning

## 7. Per-Class F1 / Precision / Recall Bar Chart

Visualises per-class performance. The red dashed line marks the recall warning
threshold (0.60). Any class below this will print a WARNING message.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Compute per-class precision, recall, F1, and support (sample count)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, labels=range(len(le.classes_))
)

metrics_df = pd.DataFrame({
    'Class'    : le.classes_,
    'Precision': prec,
    'Recall'   : rec,
    'F1'       : f1,
})

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(le.classes_))
w = 0.26   # Bar width — 3 bars per class

bars_p = ax.bar(x - w, prec, w, label='Precision', color='#378ADD')
bars_r = ax.bar(x,     rec,  w, label='Recall',    color='#1D9E75')
bars_f = ax.bar(x + w, f1,   w, label='F1',        color='#D85A30')

# Red dashed line marks the recall warning threshold
ax.axhline(RECALL_WARNING_THRESHOLD, color='red', linestyle='--',
           linewidth=1, label=f'Recall warning ({RECALL_WARNING_THRESHOLD})')

ax.set_xticks(x)
ax.set_xticklabels(le.classes_, rotation=20)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Per-class Precision / Recall / F1')
ax.legend()
plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

# Flag any class with recall below the warning threshold
weak = metrics_df[metrics_df['Recall'] < RECALL_WARNING_THRESHOLD]
if len(weak) > 0:
    print(f'\nWARNING — classes below recall threshold ({RECALL_WARNING_THRESHOLD}):')
    for _, row in weak.iterrows():
        print(f'  {row["Class"]:<15} recall={row["Recall"]:.3f}')
    print('  Consider reviewing feature separability for these classes.')
else:
    print(f'All classes above recall threshold ({RECALL_WARNING_THRESHOLD}).')

## 8. Confusion Matrix

Two views:
- **Raw counts** — shows the absolute number of correct and incorrect predictions
- **Normalized** — each row divided by its total = shows recall per class directly
  (diagonal = recall, off-diagonal = what the model confused each class with)

The most operationally dangerous error for an IDS is classifying an attack as BENIGN
(false negative). The normalized matrix reveals this clearly.

In [ ]:
cm = confusion_matrix(y_test, y_pred)   # Shape: (6, 6)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: raw counts — useful for seeing where errors concentrate in absolute terms
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Confusion matrix — raw counts')
axes[0].tick_params(axis='x', rotation=30)

# Right: normalized — each row sums to 1.0
# Diagonal value = recall for that class
# e.g. if DoS row shows 0.996 on diagonal → 99.6% of real DoS flows were correctly detected
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm.round(3), display_labels=le.classes_).plot(
    ax=axes[1], colorbar=False, cmap='Blues'
)
axes[1].set_title('Confusion matrix — normalized by true label')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the most common misclassification per class
print('Top misclassifications:')
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
for true_cls in le.classes_:
    row = cm_df.loc[true_cls].copy()
    row[true_cls] = 0   # Zero out the diagonal (correct predictions)
    if row.sum() > 0:
        worst_pred = row.idxmax()   # Which class was most often confused with this one?
        print(f'  True={true_cls:<15} → predicted as {worst_pred:<15} ({row[worst_pred]:,} times)')

## 9. Feature Importance

Random Forest computes feature importance as the **mean decrease in Gini impurity**
across all trees. Features that consistently reduce impurity the most are the most
important for separating the 6 classes.

**Expected finding:** `destination_port` should be the top feature (~21%),
consistent with the SHAP analysis of Arya et al. [1] who also found port number
to be the dominant discriminating feature on CICIDS2017. Byte-volume features
should dominate the upper rankings while TCP flags rank lowest.

In [ ]:
# rf.feature_importances_ is an array of shape (19,) — one importance per feature
importances = pd.Series(rf.feature_importances_, index=feature_contract)
importances = importances.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
# Highlight features above mean importance in orange, others in grey
colors = ['#D85A30' if imp > importances.mean() else '#B4B2A9'
          for imp in importances.values]
ax.barh(importances.index[::-1], importances.values[::-1], color=colors[::-1])
ax.axvline(importances.mean(), color='red', linestyle='--',
           linewidth=1, label='Mean importance')
ax.set_xlabel('Mean decrease in impurity (Gini)')
ax.set_title('Feature importance — 19 Suricata features')
ax.legend()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Feature importances (ranked):')
for feat, imp in importances.items():
    bar = '#' * int(imp * 500)   # ASCII bar proportional to importance
    print(f'  {feat:<22} {imp:.4f}  {bar}')

## 10. DoS Detection Deep-Dive

DoS is expected to be the strongest class because its features are extreme
and distinctive: very high packet counts, large byte volumes, and asymmetric ratios.

This cell specifically checks how many DoS flows are misclassified as BENIGN —
the most dangerous error for a volumetric attack detector.

In [ ]:
dos_idx    = list(le.classes_).index('DoS')
benign_idx = list(le.classes_).index('BENIGN')

# Isolate only the test flows that are truly DoS
dos_mask = (y_test == dos_idx)
dos_pred = y_pred[dos_mask]   # Predictions for just the DoS subset

dos_recall = (dos_pred == dos_idx).sum() / dos_mask.sum()   # Correct DoS predictions
dos_missed = (dos_pred == benign_idx).sum()                  # DoS flows missed as BENIGN
dos_total  = dos_mask.sum()

print('DoS detection summary:')
print(f'  Total DoS test samples : {dos_total:,}')
print(f'  Correctly detected     : {(dos_pred == dos_idx).sum():,}  ({dos_recall*100:.2f}%)')
print(f'  Missed as BENIGN       : {dos_missed:,}  ({dos_missed/dos_total*100:.2f}%)')
print(f'  Misclassified as other : {((dos_pred != dos_idx) & (dos_pred != benign_idx)).sum():,}')

# Recall below 0.95 for DoS would be surprising given the strong volumetric features
if dos_recall < 0.95:
    print('\nWARNING: DoS recall below 0.95 — unexpected given feature set.')
    print('Check: are flow_bytes_per_sec and flow_duration computed correctly?')
else:
    print(f'\nDoS recall is strong ({dos_recall:.4f}) — as expected.')

## 11. Probability Calibration Check

Random Forest's `predict_proba()` outputs class probabilities, but these are not
always perfectly calibrated (they tend to be too extreme — pushed toward 0 and 1).

This cell shows the distribution of maximum predicted probabilities across all test
flows, and shows what happens to accuracy and coverage when we require higher confidence
before accepting a prediction. This motivates the per-class threshold approach in Step 12.

In [ ]:
# max predicted probability for each test flow
# High values = model is very confident
# Low values = model is uncertain (prime candidates for threshold filtering)
max_proba = y_pred_proba.max(axis=1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(max_proba, bins=50, color='#378ADD', edgecolor='none', alpha=0.8)
ax.set_xlabel('Max predicted probability')
ax.set_ylabel('Count')
ax.set_title('Confidence distribution — how certain is the model?')
plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Show the trade-off: requiring higher confidence improves accuracy
# but reduces coverage (some flows get suppressed as BENIGN)
thresholds = [0.5, 0.7, 0.8, 0.9]
print('Predictions above confidence threshold:')
print(f'{"Threshold":<12} {"Samples kept":>14} {"% of total":>12} {"Accuracy on kept":>18}')
print('-' * 60)
for t in thresholds:
    mask = max_proba >= t
    kept = mask.sum()
    acc  = (y_pred[mask] == y_test.values[mask]).mean() if kept > 0 else 0
    print(f'{t:<12.1f} {kept:>14,} {kept/len(y_pred)*100:>11.1f}% {acc:>17.4f}')

print('\nNote: for Suricata deployment, a threshold of 0.7-0.8 balances')
print('coverage vs false alarms. Tune per-class in Step 12.')

## 12. Per-Class Threshold Tuning

**The problem:**
The classification report showed that Web Attack (precision=0.125) and Bot
(precision=0.290) generate many false positives — BENIGN flows are being flagged
as attacks. Their recall is fine (>89%), so the model detects real attacks but
fires too broadly at low confidence.

**The solution: per-class confidence thresholds**
Instead of always taking the `argmax` class, we require a minimum confidence
(probability) for each attack class before raising an alert. For a class like
Web Attack where precision is low, we require high confidence (0.84) before
flagging a flow. Classes with already-good precision keep lower thresholds.

**How it works at inference time:**
```
proba = model.predict_proba(flow)
raw_class = argmax(proba)
if proba[raw_class] >= threshold[raw_class]:
    alert(raw_class)
else:
    classify_as_BENIGN()  # Suppressed — below threshold
```

**Tuning method:** sweep threshold from 0.30 to 0.99 in steps of 0.01.
For each threshold value, compute precision, recall, and F1 for that class.
Select the threshold that maximises F1 for each class independently.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Sweep of threshold values to evaluate
THRESHOLDS = np.arange(0.30, 0.99, 0.01)
benign_idx  = list(le.classes_).index('BENIGN')

per_class_curves = {}   # Stores {class_name: [{threshold, precision, recall, f1}, ...]}

for cls_idx, cls_name in enumerate(le.classes_):
    if cls_name == 'BENIGN':
        continue   # No threshold needed for BENIGN — it is the fallback class
    curves = []
    for t in THRESHOLDS:
        # For each threshold: predict cls_name only if P(cls_name) >= t
        # Otherwise fall back to the argmax of the remaining predictions
        cls_proba = y_pred_proba[:, cls_idx]
        y_thresh  = np.where(cls_proba >= t, cls_idx, y_pred)

        # For this class: only count as a true positive if we actually predict it
        tp_mask = (y_thresh == cls_idx)
        if tp_mask.sum() == 0:
            # No predictions for this class at this threshold — skip
            continue

        prec = precision_score(y_test, y_thresh, labels=[cls_idx], average='micro', zero_division=0)
        rec  = recall_score(   y_test, y_thresh, labels=[cls_idx], average='micro', zero_division=0)
        f1   = f1_score(       y_test, y_thresh, labels=[cls_idx], average='micro', zero_division=0)

        curves.append({'threshold': round(t, 2), 'precision': prec, 'recall': rec, 'f1': f1})

    per_class_curves[cls_name] = curves

print('Threshold sweep complete.')
print(f'Classes tuned: {list(per_class_curves.keys())}')

In [ ]:
# Plot precision-recall-F1 curves for each attack class
# The vertical dashed line shows the selected optimal threshold
classes_to_tune = [c for c in le.classes_ if c != 'BENIGN']
n = len(classes_to_tune)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, cls_name in enumerate(classes_to_tune):
    ax     = axes[i]
    curves = per_class_curves[cls_name]
    ts     = [c['threshold'] for c in curves]
    precs  = [c['precision'] for c in curves]
    recs   = [c['recall']    for c in curves]
    f1s    = [c['f1']        for c in curves]

    ax.plot(ts, precs, color='#378ADD', label='Precision', linewidth=1.5)
    ax.plot(ts, recs,  color='#1D9E75', label='Recall',    linewidth=1.5)
    ax.plot(ts, f1s,   color='#D85A30', label='F1',        linewidth=2.0)

    # Mark the threshold that maximises F1
    best = max(curves, key=lambda x: x['f1'])
    ax.axvline(best['threshold'], color='black', linestyle='--', linewidth=1,
               label=f"θ={best['threshold']:.2f}")

    ax.set_title(f'{cls_name}  (best θ={best["threshold"]:.2f}, F1={best["f1"]:.3f})')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

plt.suptitle('Per-class Precision / Recall / F1 vs Threshold', fontsize=13)
plt.tight_layout()
plt.savefig('threshold_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Select the threshold that maximised F1 for each class
optimal_thresholds = {'BENIGN': 0.0}   # BENIGN has no threshold — it is the fallback

for cls_name, curves in per_class_curves.items():
    best = max(curves, key=lambda x: x['f1'])
    optimal_thresholds[cls_name] = best['threshold']

print('Optimal per-class thresholds (maximising F1):')
print(f'{"Class":<15} {"Threshold":>10} {"Precision":>10} {"Recall":>10} {"F1":>8}')
print('-' * 58)

# Apply ALL class thresholds simultaneously to generate the final adjusted predictions
# For each flow: keep the argmax prediction only if its probability >= that class threshold
# Otherwise reassign to BENIGN
y_pred_adjusted = y_pred.copy()
for cls_name, curves in per_class_curves.items():
    cls_idx = list(le.classes_).index(cls_name)
    t       = optimal_thresholds[cls_name]
    best    = max(curves, key=lambda x: x['f1'])

    # Where the model predicted cls_name but with confidence below threshold → reclassify as BENIGN
    low_conf_mask = (y_pred == cls_idx) & (y_pred_proba[:, cls_idx] < t)
    y_pred_adjusted[low_conf_mask] = benign_idx

    print(f'{cls_name:<15} {t:>10.2f} {best["precision"]:>10.3f} {best["recall"]:>10.3f} {best["f1"]:>8.3f}')

# Final macro F1 after threshold tuning
macro_f1_tuned = f1_score(y_test, y_pred_adjusted, average='macro')
print(f'\nMacro F1 before tuning: {macro_f1:.4f}')
print(f'Macro F1 after tuning:  {macro_f1_tuned:.4f}  (+{macro_f1_tuned - macro_f1:.3f})')
# Expected improvement: +0.039 (from 0.771 to 0.810)

In [ ]:
# Visualise the precision improvement per class before and after threshold tuning
from sklearn.metrics import precision_recall_fscore_support

prec_b, rec_b, f1_b, _ = precision_recall_fscore_support(y_test, y_pred,          labels=range(len(le.classes_)))
prec_a, rec_a, f1_a, _ = precision_recall_fscore_support(y_test, y_pred_adjusted, labels=range(len(le.classes_)))

x  = np.arange(len(le.classes_))
w  = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision comparison — main improvement area
axes[0].bar(x - w/2, prec_b, w, label='Before tuning', color='#B4B2A9')
axes[0].bar(x + w/2, prec_a, w, label='After tuning',  color='#378ADD')
axes[0].set_xticks(x)
axes[0].set_xticklabels(le.classes_, rotation=20)
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Precision: Before vs After threshold tuning')
axes[0].legend()

# F1 comparison — shows that precision improved without sacrificing too much recall
axes[1].bar(x - w/2, f1_b, w, label='Before tuning', color='#B4B2A9')
axes[1].bar(x + w/2, f1_a, w, label='After tuning',  color='#D85A30')
axes[1].set_xticks(x)
axes[1].set_xticklabels(le.classes_, rotation=20)
axes[1].set_ylim(0, 1.05)
axes[1].set_title('F1: Before vs After threshold tuning')
axes[1].legend()

plt.tight_layout()
plt.savefig('threshold_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the final classification report with thresholds applied
print('\nFINAL CLASSIFICATION REPORT (with per-class thresholds):')
print('=' * 65)
print(classification_report(y_test, y_pred_adjusted, target_names=le.classes_, digits=3))

In [ ]:
# Save thresholds to JSON — loaded by detector.py at runtime
# Maps class name → minimum confidence probability required to raise an alert
# If P(predicted_class) < threshold → classify flow as BENIGN instead

thresholds_export = {
    cls: float(round(optimal_thresholds.get(cls, 0.5), 2))
    for cls in le.classes_
}

with open('class_thresholds.json', 'w') as f:
    json.dump(thresholds_export, f, indent=2)

print('Saved: class_thresholds.json')
print()
print('Contents:')
for cls, t in thresholds_export.items():
    bar = '#' * int(t * 30)   # ASCII bar proportional to threshold value
    print(f'  {cls:<15} {t:.2f}  {bar}')
print()
print('How detector.py uses these thresholds:')
print('  raw_class = argmax(predict_proba(flow))')
print('  if proba[raw_class] >= threshold[raw_class]:')
print('      → alert as ATTACK')
print('  else:')
print('      → suppress as BENIGN (log as SUPPRESSED)')

## 13. Save Model & Artifacts

In [ ]:
# ── Save the trained model ────────────────────────────────────────────────────
# compress=3 reduces file size from ~250MB to ~32MB with minimal speed cost
joblib.dump(rf, 'random_forest_ids.pkl', compress=3)

# ── Save training results summary ─────────────────────────────────────────────
# This JSON is loaded by detector.py at startup to display model metrics
results = {
    'model'      : 'RandomForestClassifier',
    'n_features' : int(rf.n_features_in_),
    'n_classes'  : int(len(le.classes_)),
    'classes'    : list(le.classes_),
    'features'   : feature_contract,
    'rf_params'  : {k: str(v) for k, v in RF_PARAMS.items()},
    'metrics': {
        'macro_f1'       : round(float(macro_f1_tuned), 4),   # After threshold tuning
        'macro_precision': round(float(macro_prec), 4),
        'macro_recall'   : round(float(macro_recall), 4),
        'weighted_f1'    : round(float(weighted_f1), 4),
    },
    'per_class': {
        cls: {
            'precision': round(float(prec_a[i]), 4),
            'recall'   : round(float(rec_a[i]),  4),
            'f1'       : round(float(f1_a[i]),   4),
        }
        for i, cls in enumerate(le.classes_)
    },
    'feature_importances': {
        feat: round(float(imp), 6)
        for feat, imp in importances.items()
    },
}

with open('training_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Saved:')
print('  random_forest_ids.pkl    — trained model (200 trees, 32MB)')
print('  training_results.json    — full metrics and feature importances')
print()
print('All deployment artifacts:')
print('  random_forest_ids.pkl    ← loaded by detector.py')
print('  label_encoder.pkl        ← loaded by detector.py')
print('  feature_contract.pkl     ← loaded by detector.py')
print('  median_imputation.json   ← loaded by detector.py')
print('  class_thresholds.json    ← loaded by detector.py')
print()
print(f'Training complete — Macro F1 (with thresholds): {macro_f1_tuned:.4f}')